# val_4cases demo

Este notebook replica la idea de `non_iid_wrapper_demo`, pero mostrando el split `val_4cases`:
un subconjunto de validación donde cada muestra puede participar en al menos un patrón composicional de 4 casos.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from omegaconf import OmegaConf

sys.path.append("..")

from visgen.datasets import (
    Cars3D,
    DSprites,
    IRAVEN,
    MPI3D,
    Shapes3D,
    make_validation_subset as split_make_validation_subset,
)
from visgen.datasets.non_iid import NonIIDWrapper, subset_with_four_case_support
from visgen.utils.general import register_resolvers

register_resolvers()


In [ ]:
DATASET_CONFIG_DIR = Path("../configs/datasets")
# Ajusta esta ruta para apuntar a la carpeta base donde están tus datasets.
# Si los paths en los YAML son absolutos, se respetan tal cual.
DATASETS_BASE_PATH = Path("../data")

DATASET_CLASSES = {
    "dsprites": DSprites,
    "mpi3d": MPI3D,
    "shapes3d": Shapes3D,
    "cars3d": Cars3D,
    "iraven": IRAVEN,
}


def load_training_cfg(name, datasets_base_path=DATASETS_BASE_PATH):
    cfg = OmegaConf.load(DATASET_CONFIG_DIR / f"{name}.yml").data.training
    raw_path = Path(cfg.path)

    if not raw_path.is_absolute():
        try:
            relative_to_data = raw_path.relative_to("data")
        except ValueError:
            relative_to_data = raw_path
        cfg.path = str(Path(datasets_base_path) / relative_to_data)

    return cfg


def make_validation_subset(name, seed=0, datasets_base_path=DATASETS_BASE_PATH):
    cfg = load_training_cfg(name, datasets_base_path=datasets_base_path)
    dataset = DATASET_CLASSES[name](**cfg)
    _, val_data = split_make_validation_subset(
        dataset,
        val_fraction=cfg.val_fraction,
        seed=seed,
        num_ood_val=cfg.num_ood_val if "num_ood_val" in cfg else 1,
    )
    return cfg, val_data


In [ ]:
def plot_quad(images, targets, title):
    fig, axes = plt.subplots(1, 4, figsize=(12, 3))
    fig.suptitle(title)
    for idx, ax in enumerate(axes):
        img = images[idx]
        if torch.is_tensor(img):
            img = img.detach().cpu().numpy()
        if img.ndim == 3 and img.shape[0] in (1, 3):
            img = np.moveaxis(img, 0, -1)
        ax.imshow(img.squeeze(), cmap="gray")
        ax.axis("off")
        ax.set_title(str(np.asarray(targets[idx]).tolist()))
    plt.tight_layout()


In [ ]:
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

for name in DATASET_CLASSES:
    print(f"\nDataset: {name}")
    cfg, val_data = make_validation_subset(name, seed=SEED)
    allowed_attributes = list(cfg.targets) if cfg.targets else [attr.name for attr in cfg.attributes]
    shared_other_attributes = True
    if "non_iid" in cfg and cfg.non_iid is not None and not isinstance(cfg.non_iid, str):
        shared_other_attributes = cfg.non_iid.get("shared_other_attributes", True)

    val_4cases = subset_with_four_case_support(
        val_data,
        allowed_attributes=allowed_attributes,
        shared_other_attributes=shared_other_attributes,
    )
    print(f"validation size: {len(val_data)}")
    print(f"val_4cases size: {len(val_4cases)}")

    wrapper = NonIIDWrapper(
        val_4cases,
        shared_other_attributes=shared_other_attributes,
        seed=SEED,
        allowed_attributes=allowed_attributes,
    )
    images, targets = wrapper[0]
    plot_quad(images, targets, f"{name}: ejemplo de val_4cases")
